# Compute 3D cube corners (Unity SOLO)
Given a Unity SOLO `frame_data.json`, extract the first 3D bounding box pose and compute oriented box corners,
optionally applying an additional local transform and custom edge lengths.

In [1]:
# Imports
import os, json, math
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import numpy as np

In [2]:
# Helper functions (ported from compute_cube_coords.py)
def quat_to_matrix(qx: float, qy: float, qz: float, qw: float) -> np.ndarray:
    q = np.array([qx, qy, qz, qw], dtype=float)
    norm = np.linalg.norm(q)
    if norm == 0: raise ValueError('Zero-length quaternion')
    q = q / norm; x, y, z, w = q
    xx, yy, zz = x*x, y*y, z*z
    xy, xz, yz = x*y, x*z, y*z
    wx, wy, wz = w*x, w*y, w*z
    R = np.array([[1-2*(yy+zz), 2*(xy-wz), 2*(xz+wy)],
                  [2*(xy+wz), 1-2*(xx+zz), 2*(yz-wx)],
                  [2*(xz-wy), 2*(yz+wx), 1-2*(xx+yy)]], dtype=float)
    return R

def euler_xyz_to_matrix(rx_deg: float, ry_deg: float, rz_deg: float) -> np.ndarray:
    rx, ry, rz = map(math.radians, (rx_deg, ry_deg, rz_deg))
    Rx = np.array([[1,0,0],[0, math.cos(rx), -math.sin(rx)],[0, math.sin(rx), math.cos(rx)]], dtype=float)
    Ry = np.array([[ math.cos(ry),0, math.sin(ry)],[0,1,0],[-math.sin(ry),0, math.cos(ry)]], dtype=float)
    Rz = np.array([[math.cos(rz), -math.sin(rz),0],[math.sin(rz), math.cos(rz),0],[0,0,1]], dtype=float)
    return Rz @ Ry @ Rx

def find_first_bbox(values: List[Dict]) -> Optional[Dict]:
    for v in values:
        if all(k in v for k in ('translation','rotation')):
            return v
    return None

def extract_pose_from_frame_json(frame_json: Dict, camera_id: Optional[str] = None) -> Tuple[np.ndarray, np.ndarray]:
    captures = frame_json.get('captures', [])
    for cap in captures:
        if camera_id is not None and cap.get('id') != camera_id: continue
        anns = cap.get('annotations', [])
        for ann in anns:
            if ann.get('@type','').endswith('BoundingBox3DAnnotation'):
                values = ann.get('values', [])
                bbox = find_first_bbox(values)
                if bbox is None: continue
                t = np.array(bbox['translation'], dtype=float)
                q = np.array(bbox['rotation'], dtype=float)
                if q.shape[0] != 4: raise ValueError('Expected quaternion [x,y,z,w]')
                R = quat_to_matrix(q[0], q[1], q[2], q[3])
                return t, R
    raise ValueError('No BoundingBox3DAnnotation with translation/rotation found')

def compute_corners_from_corner(base_corner_world: np.ndarray, R_axes_world: np.ndarray, size_xyz: Tuple[float,float,float]) -> Dict[str, List[float]]:
    Lx, Ly, Lz = size_xyz
    ux, uy, uz = R_axes_world[:,0], R_axes_world[:,1], R_axes_world[:,2]
    corners = {}
    for a in (0,1):
        for b in (0,1):
            for c in (0,1):
                name = f'c{a}{b}{c}'
                offset = (a*Lx)*ux + (b*Ly)*uy + (c*Lz)*uz
                p = base_corner_world + offset
                corners[name] = [float(p[0]), float(p[1]), float(p[2])]
    return corners

In [3]:
# Parameters
frame_json_path = r'./image_sample/step0.frame_data.json'  # TODO: set to your frame_data.json
camera_id = None  # e.g., 'camera1' or leave None
offset_translation = [9.0, -5.7, 11.9319]
offset_rotation = [-90.0, 90.0, -90.0]  # XYZ degrees
size_xyz = [10.4, 6.0, 2.8]
out_json = None  # e.g., './output/corners.json'
print('Using frame_json_path:', frame_json_path)

Using frame_json_path: ./image_sample/step0.frame_data.json


In [ ]:
# Run
with open(frame_json_path, 'r', encoding='utf-8') as f:
    frame = json.load(f)

center_t, R_box = extract_pose_from_frame_json(frame, camera_id=camera_id)
R_off = euler_xyz_to_matrix(*offset_rotation)
t_off = np.array(offset_translation, dtype=float)
base_corner_world = center_t + R_box @ (R_off @ t_off)
R_axes_world = R_box @ R_off
corners = compute_corners_from_corner(base_corner_world, R_axes_world, tuple(size_xyz))

result = {
    'frame_json': frame_json_path,
    'camera_id': camera_id,
    'center': [float(center_t[0]), float(center_t[1]), float(center_t[2])],
    'base_corner': [float(base_corner_world[0]), float(base_corner_world[1]), float(base_corner_world[2])],
    'size': list(map(float, size_xyz)),
    'offset_translation': list(map(float, offset_translation)),
    'offset_rotation_xyz_deg': list(map(float, offset_rotation)),
    'corners': corners
}
print(json.dumps(result, indent=2))

if out_json is not None:
    os.makedirs(os.path.dirname(out_json), exist_ok=True)
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2)
    print(f'Wrote corners to: {out_json}')